In [1]:
# ============================================================
# 10. RANDOM FOREST REGRESSION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 10.1 LOAD DATA
# ============================================================

SCORED_PATH = r"C:/Users/muskan/OneDrive/Desktop/effisense/data/processed/mouse_productivity_scores_reduced.csv"

df = pd.read_csv(SCORED_PATH)

TARGET = "productivity_score"

BEHAVIORAL_FEATURES = [
    "idle_percentage",
    "pause_count",
    "mean_pause",
    "mean_velocity",
    "total_distance",
    "mean_distance",
    "velocity_cv",
    "velocity_entropy",
    "smoothness",
    "mean_acceleration",
    "std_acceleration",
    "mean_direction_change",
    "direction_entropy",
    "num_events"
]


# ============================================================
# 10.2 PREPARE DATA
# ============================================================

X = df[BEHAVIORAL_FEATURES].copy()
y = df[TARGET].copy()

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())


# ============================================================
# 10.3 SESSION-BASED SPLIT
# ============================================================

groups = (
    df["user"].astype(str)
    + "_"
    + df["session"].astype(str)
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

overlap = set(groups_train).intersection(
    set(groups_test)
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Train sessions:", groups_train.nunique())
print("Test sessions :", groups_test.nunique())
print("Session overlap:", len(overlap))


# ============================================================
# 10.4 RANDOM FOREST
# ============================================================

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)


# ============================================================
# 10.5 PREDICTIONS
# ============================================================

rf_raw_pred = rf_model.predict(X_test)

# BPI is defined from 0–100
rf_pred = np.clip(rf_raw_pred, 0, 100)


# ============================================================
# 10.6 EVALUATION
# ============================================================

rf_mae = mean_absolute_error(y_test, rf_pred)

rf_mse = mean_squared_error(
    y_test,
    rf_pred
)

rf_rmse = np.sqrt(rf_mse)

rf_r2 = r2_score(
    y_test,
    rf_pred
)


print("\n==============================")
print("RANDOM FOREST RESULTS")
print("==============================")

print(f"MAE  : {rf_mae:.4f}")
print(f"MSE  : {rf_mse:.4f}")
print(f"RMSE : {rf_rmse:.4f}")
print(f"R²   : {rf_r2:.4f}")


# ============================================================
# 10.7 SAMPLE PREDICTIONS
# ============================================================

rf_comparison = pd.DataFrame({
    "actual_score": y_test.values,
    "predicted_score": rf_pred
})

print("\nSample predictions:")
print(rf_comparison.head(10))


# ============================================================
# 10.8 FEATURE IMPORTANCE
# ============================================================

rf_importance = pd.DataFrame({
    "feature": BEHAVIORAL_FEATURES,
    "importance": rf_model.feature_importances_
})

rf_importance = rf_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

print("\n==============================")
print("RANDOM FOREST FEATURE IMPORTANCE")
print("==============================")

print(
    rf_importance.to_string(index=False)
)

X_train: (7980, 14)
X_test : (2020, 14)
Train sessions: 72
Test sessions : 19
Session overlap: 0

RANDOM FOREST RESULTS
MAE  : 1.1689
MSE  : 2.2381
RMSE : 1.4960
R²   : 0.9971

Sample predictions:
   actual_score  predicted_score
0     71.042882        69.867434
1     49.248414        46.211858
2     50.297183        47.491514
3     51.618271        48.246698
4     33.251916        30.279159
5     35.827960        32.057611
6     34.413828        30.969921
7     34.720532        32.741710
8     34.859078        33.132147
9     49.445956        46.728247

RANDOM FOREST FEATURE IMPORTANCE
              feature  importance
      idle_percentage    0.666673
          velocity_cv    0.181977
           smoothness    0.131237
     velocity_entropy    0.008141
       total_distance    0.003955
        mean_distance    0.002414
           mean_pause    0.001580
          pause_count    0.001355
        mean_velocity    0.001326
mean_direction_change    0.000363
    direction_entropy    0.00033

In [2]:
# ============================================================
# 11. 5-FOLD SESSION-LEVEL CROSS-VALIDATION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 11.1 LOAD DATA
# ============================================================

SCORED_PATH = r"C:/Users/muskan/OneDrive/Desktop/effisense/data/processed/mouse_productivity_scores_reduced.csv"

df = pd.read_csv(SCORED_PATH)

TARGET = "productivity_score"

BEHAVIORAL_FEATURES = [
    "idle_percentage",
    "pause_count",
    "mean_pause",
    "mean_velocity",
    "total_distance",
    "mean_distance",
    "velocity_cv",
    "velocity_entropy",
    "smoothness",
    "mean_acceleration",
    "std_acceleration",
    "mean_direction_change",
    "direction_entropy",
    "num_events"
]


# ============================================================
# 11.2 PREPARE DATA
# ============================================================

X = df[BEHAVIORAL_FEATURES].copy()
y = df[TARGET].copy()

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())

# IMPORTANT:
# Session identity must include the user because session IDs
# restart for different users.
groups = (
    df["user"].astype(str)
    + "_"
    + df["session"].astype(str)
)


# ============================================================
# 11.3 DEFINE MODELS
# ============================================================

linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

rf_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)


# ============================================================
# 11.4 GROUP K-FOLD
# ============================================================

gkf = GroupKFold(n_splits=5)


linear_results = []
rf_results = []


# ============================================================
# 11.5 CROSS-VALIDATION
# ============================================================

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    train_groups = groups.iloc[train_idx]
    test_groups = groups.iloc[test_idx]

    overlap = set(train_groups).intersection(
        set(test_groups)
    )

    print(f"\n========== FOLD {fold} ==========")
    print("Train windows :", len(train_idx))
    print("Test windows  :", len(test_idx))
    print("Train sessions:", train_groups.nunique())
    print("Test sessions :", test_groups.nunique())
    print("Session overlap:", len(overlap))


    # --------------------------------------------------------
    # LINEAR REGRESSION
    # --------------------------------------------------------

    linear_model.fit(X_train, y_train)

    linear_pred = linear_model.predict(X_test)

    # BPI valid range
    linear_pred = np.clip(
        linear_pred,
        0,
        100
    )

    linear_mae = mean_absolute_error(
        y_test,
        linear_pred
    )

    linear_mse = mean_squared_error(
        y_test,
        linear_pred
    )

    linear_rmse = np.sqrt(linear_mse)

    linear_r2 = r2_score(
        y_test,
        linear_pred
    )

    linear_results.append({
        "fold": fold,
        "MAE": linear_mae,
        "MSE": linear_mse,
        "RMSE": linear_rmse,
        "R2": linear_r2
    })


    # --------------------------------------------------------
    # RANDOM FOREST
    # --------------------------------------------------------

    rf_model.fit(X_train, y_train)

    rf_pred = rf_model.predict(X_test)

    # BPI valid range
    rf_pred = np.clip(
        rf_pred,
        0,
        100
    )

    rf_mae = mean_absolute_error(
        y_test,
        rf_pred
    )

    rf_mse = mean_squared_error(
        y_test,
        rf_pred
    )

    rf_rmse = np.sqrt(rf_mse)

    rf_r2 = r2_score(
        y_test,
        rf_pred
    )

    rf_results.append({
        "fold": fold,
        "MAE": rf_mae,
        "MSE": rf_mse,
        "RMSE": rf_rmse,
        "R2": rf_r2
    })


# ============================================================
# 11.6 RESULTS
# ============================================================

linear_results_df = pd.DataFrame(
    linear_results
)

rf_results_df = pd.DataFrame(
    rf_results
)


print("\n\n========================================")
print("LINEAR REGRESSION - 5 FOLD RESULTS")
print("========================================")

print(
    linear_results_df.to_string(
        index=False
    )
)


print("\n\n========================================")
print("RANDOM FOREST - 5 FOLD RESULTS")
print("========================================")

print(
    rf_results_df.to_string(
        index=False
    )
)


# ============================================================
# 11.7 MEAN ± STANDARD DEVIATION
# ============================================================

print("\n\n========================================")
print("FINAL CROSS-VALIDATION COMPARISON")
print("========================================")

for metric in ["MAE", "MSE", "RMSE", "R2"]:

    linear_mean = linear_results_df[metric].mean()
    linear_std = linear_results_df[metric].std()

    rf_mean = rf_results_df[metric].mean()
    rf_std = rf_results_df[metric].std()

    print(f"\n{metric}")

    print(
        f"Linear Regression : "
        f"{linear_mean:.4f} ± {linear_std:.4f}"
    )

    print(
        f"Random Forest     : "
        f"{rf_mean:.4f} ± {rf_std:.4f}"
    )


========== FOLD 1 ==========
Train windows : 8000
Test windows  : 2000
Train sessions: 74
Test sessions : 17
Session overlap: 0

========== FOLD 2 ==========
Train windows : 8000
Test windows  : 2000
Train sessions: 73
Test sessions : 18
Session overlap: 0

========== FOLD 3 ==========
Train windows : 8000
Test windows  : 2000
Train sessions: 73
Test sessions : 18
Session overlap: 0

========== FOLD 4 ==========
Train windows : 8000
Test windows  : 2000
Train sessions: 72
Test sessions : 19
Session overlap: 0

========== FOLD 5 ==========
Train windows : 8000
Test windows  : 2000
Train sessions: 72
Test sessions : 19
Session overlap: 0


LINEAR REGRESSION - 5 FOLD RESULTS
 fold      MAE      MSE     RMSE       R2
    1 1.829954 6.255034 2.501007 0.989808
    2 1.938494 6.229155 2.495828 0.989668
    3 1.348840 3.665949 1.914667 0.989751
    4 1.671508 5.073875 2.252526 0.991387
    5 1.635356 4.335150 2.082102 0.992359


RANDOM FOREST - 5 FOLD RESULTS
 fold      MAE      MSE     RMSE 

### FEATURE IMPORTANCE

In [3]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold


# ==============================
# 1. Load reduced dataset
# ==============================

DATA_PATH = r"C:\\Users\\muskan\\OneDrive\\Desktop\\effisense\\data\\processed\\mouse_productivity_scores_reduced.csv"

df = pd.read_csv(DATA_PATH)


# ==============================
# 2. Features and target
# ==============================

REDUCED_FEATURES = [
    "idle_percentage",
    "pause_count",
    "mean_pause",
    "mean_velocity",
    "total_distance",
    "mean_distance",
    "velocity_cv",
    "velocity_entropy",
    "smoothness",
    "mean_acceleration",
    "std_acceleration",
    "mean_direction_change",
    "direction_entropy",
    "num_events"
]

TARGET = "productivity_score"


X = df[REDUCED_FEATURES].copy()
y = df[TARGET].copy()

# Safety: handle missing values
X = X.fillna(X.median())


# ==============================
# 3. Create unique session groups
# ==============================

# Session numbers restart for different users,
# so session alone is NOT a unique identifier.

groups = (
    df["user"].astype(str)
    + "_"
    + df["session"].astype(str)
)


# ==============================
# 4. 5-Fold Group CV
# ==============================

gkf = GroupKFold(n_splits=5)

importance_results = []


for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]


    # ==============================
    # 5. Train Random Forest
    # ==============================

    rf = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)


    # ==============================
    # 6. Store feature importance
    # ==============================

    fold_importance = pd.DataFrame({
        "feature": REDUCED_FEATURES,
        "importance": rf.feature_importances_
    })

    fold_importance["fold"] = fold

    importance_results.append(fold_importance)


# Combine all folds
importance_df = pd.concat(
    importance_results,
    ignore_index=True
)


# ==============================
# 7. Mean ± SD across folds
# ==============================

feature_importance_summary = (
    importance_df
    .groupby("feature")["importance"]
    .agg(["mean", "std"])
    .sort_values("mean", ascending=False)
    .reset_index()
)


feature_importance_summary["rank"] = (
    feature_importance_summary["mean"]
    .rank(
        ascending=False,
        method="min"
    )
    .astype(int)
)


# ==============================
# 8. Display
# ==============================

print("\nRF Feature Importance Across 5 Session-Level Folds")
print("=" * 60)

print(
    feature_importance_summary[
        ["rank", "feature", "mean", "std"]
    ].to_string(index=False)
)


RF Feature Importance Across 5 Session-Level Folds
 rank               feature     mean      std
    1       idle_percentage 0.632921 0.148463
    2           velocity_cv 0.213245 0.124476
    3            smoothness 0.142675 0.030674
    4        total_distance 0.004105 0.000496
    5         mean_distance 0.001547 0.000377
    6           pause_count 0.001441 0.000221
    7         mean_velocity 0.001297 0.000433
    8            mean_pause 0.000890 0.000398
    9      velocity_entropy 0.000692 0.000360
   10 mean_direction_change 0.000314 0.000022
   11     direction_entropy 0.000296 0.000013
   12      std_acceleration 0.000294 0.000013
   13     mean_acceleration 0.000285 0.000020
   14            num_events 0.000000 0.000000
